In [1]:
# Step 1
import io, textwrap
import numpy as np
import CoolProp.CoolProp as CP
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Output: the cells print nothing (unless SHOW_IN_CELLS = True). Every message,
# the report, all figures and the references are written to ONE PDF in Step 15.
SHOW_IN_CELLS = False
OUT_PDF = 'Lunar_TES_sCO2_results.pdf'
RUN_LOG, REPORT = [], []

def say(*args, sep=' '):
    """Message from a cell: kept for the PDF run log (printed only if SHOW_IN_CELLS)."""
    text = sep.join(str(a) for a in args)
    RUN_LOG.extend(text.split('\n'))
    if SHOW_IN_CELLS:
        print(text)

def rep(*args, sep=' '):
    """Line of the report (Step 12): kept for the PDF (printed only if SHOW_IN_CELLS)."""
    text = sep.join(str(a) for a in args)
    REPORT.extend(text.split('\n'))
    if SHOW_IN_CELLS:
        print(text)

In [2]:
# Step 2
# Inputs: every input value is in the file INPUTS_FILE, next to this notebook.
# To run another case, edit that file (or point INPUTS_FILE to a copy of it) and run
# the notebook from Step 1. This cell reads it, checks it and logs a summary.
# Start every run with an empty log and report, so re-running the notebook from the
# inputs never repeats lines in the PDF (Step 12 also clears REPORT before it writes).
RUN_LOG.clear()
REPORT.clear()
INPUTS_FILE = 'lunar_inputs.py'
with open(INPUTS_FILE) as _f:
    exec(compile(_f.read(), INPUTS_FILE, 'exec'))
say(f"Inputs read from {INPUTS_FILE}")

In [3]:
# Step 3
# Regolith TES properties (specific heat, stored energy, conductivity, heat loss) are in
# the file REGOLITH_FILE, next to this notebook. This cell loads them.
REGOLITH_FILE = 'lunar_regolith.py'
with open(REGOLITH_FILE) as _f:
    exec(compile(_f.read(), REGOLITH_FILE, 'exec'))
say(f"Regolith properties read from {REGOLITH_FILE} ({regolith_type})")

In [4]:
# Step 4
# ---------------------------------------------------------------------
# Simple sCO2 Brayton cycle (Appendix Table A, Eq. 1-10) with pressure drops
# States: 1 compressor in, 2 compressor out / PHX in,
#         3 PHX out / turbine in, 4 turbine out / cooler in
# Pressures: P1 = P_low, P2 = P_high, P3 = P2*(1 - dp_PHX),
#            P4 = P1/(1 - dp_cooler)  (the turbine expands only to the pressure
#            that leaves P_low after the cooler/radiator)
# Isentropic efficiencies are applied to ENTHALPY (not temperature, which is only
# valid for an ideal gas with constant cp); outlet temperatures follow from (h, P):
#   compressor: h2 = h1 + (h2s - h1) / eta_comp,   T2 = T(h2, P2)      Eq. 8
#   turbine:    h4 = h3 - eta_turb * (h3 - h4s),   T4 = T(h4, P4)      Eq. 10
# T1 = compressor inlet temperature, set each hour by the radiator (Step 6).
# Results are pre-computed on a (T1, TIT) grid and stored in CYC[cycle];
# cycle_row(TIT, ct) gives all T1 values at one TIT, cycle_at(TIT, ct, T1) one point.
# Validated against Sandia SAND2012-6426 (see the validation note in the project).
# ---------------------------------------------------------------------
P_simple = dict(P1=P_low, P2=P_high, P3=P_high * (1 - dp_PHX), P4=P_low / (1 - dp_cooler))
P_phx_out = {'simple': P_simple['P3']}               # PHX outlet = turbine inlet pressure

# Speed: the compressor depends only on T1 and the turbine only on TIT, so each is
# calculated once per value and reused across the grid (same CoolProp calls, same
# numbers: the results are identical). The cache keys include every input used, so a
# changed pressure, efficiency or fluid is recalculated.
from functools import lru_cache

@lru_cache(maxsize=None)
def compressor(T1, P_in, P_out, eta, fluid):
    """Compressor (Eq. 8): isentropic efficiency on enthalpy -> (h1, h2, T2)."""
    h1 = CP.PropsSI('H', 'T', T1, 'P', P_in, fluid)
    s1 = CP.PropsSI('S', 'T', T1, 'P', P_in, fluid)
    h2s = CP.PropsSI('H', 'S', s1, 'P', P_out, fluid)
    h2 = h1 + (h2s - h1) / eta
    return h1, h2, CP.PropsSI('T', 'H', h2, 'P', P_out, fluid)

@lru_cache(maxsize=None)
def turbine(TIT, P_in, P_out, eta, fluid):
    """Turbine (Eq. 10): isentropic efficiency on enthalpy -> (h_in, h_out, T_out)."""
    h3 = CP.PropsSI('H', 'T', TIT, 'P', P_in, fluid)
    s3 = CP.PropsSI('S', 'T', TIT, 'P', P_in, fluid)
    h4s = CP.PropsSI('H', 'S', s3, 'P', P_out, fluid)
    h4 = h3 - eta * (h3 - h4s)
    return h3, h4, CP.PropsSI('T', 'H', h4, 'P', P_out, fluid)

def simple_cycle(TIT, T1=None):
    T1 = T_comp_in if T1 is None else T1
    P = P_simple
    # Compressor (Eq. 8) and turbine (Eq. 10): isentropic efficiencies on enthalpy
    h1, h2, T2 = compressor(float(T1), P['P1'], P['P2'], eta_comp, FLUID)
    h3, h4, T4 = turbine(float(TIT), P['P3'], P['P4'], eta_turb, FLUID)

    w_c = h2 - h1                 # Eq. 1 (per kg)
    q_in = h3 - h2                # Eq. 2
    w_t = h3 - h4                 # Eq. 3
    q_out = h4 - h1               # Eq. 4
    w_net = w_t - w_c             # Eq. 5
    eta = w_net / q_in            # Eq. 6
    return dict(eta=eta, w_net=w_net, w_c=w_c, w_t=w_t, q_in=q_in, q_out=q_out,
                q_rec=0.0, T1=T1, T2=T2, T4=T4, T_cool_in=T4, h_cool_in=h4, h_phx_in=h2,
                h1=h1, h2=h2, h3=h3, h4=h4,
                P=[P['P1'], P['P2'], P['P3'], P['P4']])

# --- Grids: compressor inlet T1 x turbine inlet TIT ---
T1_grid = np.arange(T_comp_in, T_comp_in_max + 0.1, 2.5)
TIT_grid = np.unique(np.concatenate([np.arange(T_comp_in + 30, TIT_top, 5.0), [TIT_max, TIT_top]]))
KEYS = ['eta', 'w_net', 'w_c', 'w_t', 'q_in', 'q_out', 'q_rec', 'h1', 'h_cool_in', 'T_cool_in',
        'h_phx_in']

def build_cycle_table(cycle_fn):
    """Pre-compute a cycle on the (T1, TIT) grid. The TES cutoff uses the coldest T1."""
    G = {k: np.full((len(T1_grid), len(TIT_grid)), np.nan) for k in KEYS}
    for i, T1 in enumerate(T1_grid):
        for j, TIT in enumerate(TIT_grid):
            try:
                r = cycle_fn(TIT, T1)
            except ValueError:            # CoolProp could not solve this point: left as NaN
                continue
            for k in KEYS:
                G[k][i, j] = r[k]
    with np.errstate(invalid='ignore'):
        ok = (G['w_net'][0] > 0) & (G['q_in'][0] > 0) & (G['eta'][0] >= eta_cycle_min)
    if not ok[-1]:
        raise RuntimeError(f"{cycle_fn.__name__}: the cycle does not work even at the top of the "
                           f"TIT grid ({TIT_grid[-1]:.0f} K); check T_TES_max, pressures and efficiencies")
    # lowest TIT at which the cycle still works (scan down from the top, stop at first failure)
    first_fail_from_top = np.argmax(~ok[::-1])
    TIT_cut = TIT_grid[len(TIT_grid) - first_fail_from_top] if not ok.all() else TIT_grid[0]
    return dict(G=G, TIT_cutoff=TIT_cut, T_TES_cutoff=max(T_TES_min, TIT_cut + dT_PHX + dT_TES_HX))   # night: gas exit T_TES - dT_TES_HX

def cycle_row(TIT, ct):
    """All cycle results versus T1 (arrays over T1_grid) at one TIT, plus a 'works' mask.
    TIT outside the grid is held at the nearest grid edge (no extrapolation)."""
    TIT = float(np.clip(TIT, TIT_grid[0], TIT_grid[-1]))
    j = int(np.clip(np.searchsorted(TIT_grid, TIT) - 1, 0, len(TIT_grid) - 2))
    w = (TIT - TIT_grid[j]) / (TIT_grid[j + 1] - TIT_grid[j])
    G = CYC[ct]['G']
    r = {k: G[k][:, j] * (1 - w) + G[k][:, j + 1] * w for k in KEYS}
    with np.errstate(invalid='ignore'):
        r['ok'] = (r['w_net'] > 0) & (r['q_in'] > 0) & (r['eta'] >= eta_cycle_min)
    return r

def cycle_at(TIT, ct='simple', T1=None):
    """Cycle results at one (TIT, T1); T1 defaults to the coldest compressor inlet."""
    T1 = T_comp_in if T1 is None else T1
    r = cycle_row(TIT, ct)
    return {k: float(np.interp(T1, T1_grid, r[k])) for k in KEYS}

say(f"Building simple-cycle map ({len(T1_grid)} x {len(TIT_grid)} points)...")
CYC = {'simple': build_cycle_table(simple_cycle)}
say(f"simple cycle: turbine {P_simple['P3']/1e6:.2f} -> {P_simple['P4']/1e6:.2f} MPa, "
    f"eta at TIT {TIT_max:.0f} K / T1 {T_comp_in:.0f} K = {cycle_at(TIT_max, 'simple')['eta']*100:.2f} %, "
    f"usable TES down to {CYC['simple']['T_TES_cutoff']:.0f} K")

In [5]:
# Step 5
# ---------------------------------------------------------------------
# Recuperated sCO2 Brayton cycle (Appendix Table A, Eq. 11-20)
# States: 1 compressor in        2 compressor out / recuperator cold in
#         3 recuperator cold out / PHX in      4 PHX out / turbine in (TIT)
#         5 turbine out / recuperator hot in   6 recuperator hot out / cooler in
# Pressures: P1 = P_low, P2 = P_high, P3 = P2(1-dp_recup), P4 = P3(1-dp_PHX),
#            P6 = P1/(1-dp_cooler), P5 = P6/(1-dp_recup)
# Compressor and turbine as in Step 4 (isentropic efficiencies on ENTHALPY, Eq. 8, 10).
# Recuperator (Eq. 11-14), counterflow:
#   q_max = the largest duty for which the hot stream stays hotter than the cold stream
#           EVERYWHERE inside the exchanger, not only at its two ends. sCO2 cp changes
#           strongly with temperature, so the closest approach can sit inside the
#           exchanger; the profile is checked at n_recup_check points and q_max is
#           reduced (bisection) if the streams would cross.
#   q_rec = eps_recup x q_max;   h3 = h2 + q_rec,  h6 = h5 - q_rec   (Q_cold = Q_hot)
#   The recuperator is always in the flow path (no bypass). If the turbine exhaust (5) is
#   colder than the compressor outlet (2), heat flows the other way, from the high-pressure
#   stream to the exhaust: q_rec < 0 with the same effectiveness, so the sCO2 reaches the
#   PHX colder and the precooler hotter. The duty passes smoothly through 0 at T5 = T2.
# The smallest temperature difference inside the exchanger (between whichever stream is
# hotter and the other) is returned as dT_pinch (K) when pinch=True (report only).
# Validated against Sandia SAND2012-6426 (see the validation note in the project).
# ---------------------------------------------------------------------
P_recup = dict(P1=P_low, P2=P_high, P3=P_high * (1 - dp_recup))
P_recup['P4'] = P_recup['P3'] * (1 - dp_PHX)                 # turbine inlet
P_recup['P6'] = P_low / (1 - dp_cooler)                      # recuperator hot out / cooler in
P_recup['P5'] = P_recup['P6'] / (1 - dp_recup)               # turbine outlet
P_phx_out['recuperated'] = P_recup['P4']
n_recup_check = 10                                           # points along the exchanger

# Speed: the recuperator crossing check (called ~22,000 times, ~94 % of this cell's time)
# reads T(h, P) from CoolProp's property tables (BICUBIC&HEOS: built once, saved on disk,
# ~300x faster, T within 0.001 K of the full equation of state). Every cycle state
# (compressor, turbine, recuperator ends) still uses the full equation of state (PropsSI).
_AS_tab = CP.AbstractState('BICUBIC&HEOS', FLUID)

def _T_hP_tab(h, p):
    out = np.empty(len(h))
    for i in range(len(h)):
        _AS_tab.update(CP.HmassP_INPUTS, h[i], p[i])
        out[i] = _AS_tab.T()
    return out

def recup_min_dT(h2, h5, q, P):
    """Smallest temperature difference, in the direction the heat flows, along a counterflow
    recuperator carrying duty q (J/kg; q > 0: exhaust -> high-pressure side, q < 0: the
    reverse). Point f runs from f = 0 (exhaust outlet h5 - q, high-pressure inlet h2) to
    f = 1 (exhaust inlet h5, high-pressure outlet h2 + q); pressures vary linearly."""
    f = np.linspace(0.0, 1.0, n_recup_check + 1)
    T_lp = _T_hP_tab(h5 - q + f * q, P['P6'] + f * (P['P5'] - P['P6']))
    T_hp = _T_hP_tab(h2 + f * q, P['P2'] + f * (P['P3'] - P['P2']))
    sgn = 1.0 if q >= 0 else -1.0
    return float(np.min(sgn * (np.asarray(T_lp) - np.asarray(T_hp))))

def recuperated_cycle(TIT, T1=None, pinch=False):
    """pinch=True also returns dT_pinch (slower; used for the report, not for the tables)."""
    T1 = T_comp_in if T1 is None else T1
    P = P_recup
    # Compressor (Eq. 8) and turbine (Eq. 10): same functions as Step 4, each calculated
    # once per T1 / TIT and reused (identical numbers)
    h1, h2, T2 = compressor(float(T1), P['P1'], P['P2'], eta_comp, FLUID)
    h4, h5, T5 = turbine(float(TIT), P['P4'], P['P5'], eta_turb, FLUID)

    # Recuperator (Eq. 11-14)
    # Largest duty from the end temperatures: the stream giving heat can at most be cooled to
    # the other stream's inlet, the stream taking heat at most heated to the other's inlet.
    dh_lp = h5 - CP.PropsSI('H', 'T', T2, 'P', P['P6'], FLUID)        # exhaust brought to T2
    dh_hp = CP.PropsSI('H', 'T', T5, 'P', P['P3'], FLUID) - h2        # HP stream brought to T5
    if T5 >= T2:                                   # normal: exhaust heats the HP stream
        q_max = max(min(dh_lp, dh_hp), 0.0)
    else:                                          # reverse: HP stream heats the exhaust
        q_max = -max(min(-dh_lp, -dh_hp), 0.0)     # q_max <= 0
    if q_max != 0 and recup_min_dT(h2, h5, q_max, P) < 0:
        # the streams would cross inside the exchanger: largest |duty| with no crossing
        lo, hi = 0.0, 1.0
        for _ in range(30):
            mid = 0.5 * (lo + hi)
            lo, hi = (mid, hi) if recup_min_dT(h2, h5, mid * q_max, P) >= 0 else (lo, mid)
        q_max = lo * q_max
    q_rec = eps_recup * q_max                            # Eq. 13: Q_cold = Q_hot
    h3 = h2 + q_rec                                      # Eq. 11
    h6 = h5 - q_rec                                      # Eq. 12
    T3 = CP.PropsSI('T', 'H', h3, 'P', P['P3'], FLUID)
    T6 = CP.PropsSI('T', 'H', h6, 'P', P['P6'], FLUID)
    dT_pinch = recup_min_dT(h2, h5, q_rec, P) if (pinch and q_rec != 0) else np.nan

    w_c = h2 - h1                 # Eq. 17 (per kg)
    q_in = h4 - h3                # Eq. 15: heat from TES after recuperation
    w_t = h4 - h5                 # Eq. 16
    q_out = h6 - h1               # Eq. 20: heat rejected after recuperation
    w_net = w_t - w_c             # Eq. 18
    eta = w_net / q_in            # Eq. 19
    return dict(eta=eta, w_net=w_net, w_c=w_c, w_t=w_t, q_in=q_in, q_out=q_out,
                q_rec=q_rec, dT_pinch=dT_pinch,
                T1=T1, T2=T2, T3=T3, T5=T5, T6=T6, T_cool_in=T6, h_cool_in=h6, h_phx_in=h3,
                h1=h1, h2=h2, h3=h3, h4=h4, h5=h5, h6=h6,
                P=[P['P1'], P['P2'], P['P3'], P['P4'], P['P5'], P['P6']])

def cycle_state_table(TIT, ct, T1=None):
    """Cycle results and (name, T, P) of every state, for the report."""
    if ct == 'recuperated':
        r = recuperated_cycle(TIT, T1, pinch=True)
        names = ['1  Compressor inlet', '2  Compressor outlet', '3  Recuperator cold out',
                 '4  Turbine inlet', '5  Turbine outlet', '6  Cooler inlet']
        temps = [r['T1'], r['T2'], r['T3'], TIT, r['T5'], r['T6']]
    else:
        r = simple_cycle(TIT, T1)
        names = ['1  Compressor inlet', '2  Compressor outlet', '3  Turbine inlet', '4  Turbine outlet']
        temps = [r['T1'], r['T2'], TIT, r['T4']]
    return r, list(zip(names, temps, r['P']))

say(f"Building recuperated-cycle map ({len(T1_grid)} x {len(TIT_grid)} points)...")
CYC['recuperated'] = build_cycle_table(recuperated_cycle)
_r = recuperated_cycle(TIT_max, T_comp_in, pinch=True)
say(f"recuperated cycle: turbine {P_recup['P4']/1e6:.2f} -> {P_recup['P5']/1e6:.2f} MPa, "
    f"eta at TIT {TIT_max:.0f} K / T1 {T_comp_in:.0f} K = {cycle_at(TIT_max, 'recuperated')['eta']*100:.2f} %, "
    f"recuperator pinch {_r['dT_pinch']:.1f} K, "
    f"usable TES down to {CYC['recuperated']['T_TES_cutoff']:.0f} K")

In [6]:
# Step 6
# Plant model (layout of Fig. 2: TES gas loop, power cycle, coolant loop, habitat, radiator)
# is in the file PLANT_FILE, next to this notebook. This cell loads it: it defines
# plant_point() and the loop/radiator functions, adds the loop flow ratios to the cycle
# maps, finds the usable TES range and sizes the radiator for the TES floor and the PHX.
PLANT_FILE = 'lunar_plant.py'
with open(PLANT_FILE) as _f:
    exec(compile(_f.read(), PLANT_FILE, 'exec'))
say(f"Plant model read from {PLANT_FILE}")

In [7]:
# Step 7
# Discharge table (night): cycle and plant efficiency, heat from the TES and TES - TIT gap
# every 25 K of TES temperature, for both cycles. The code is in the file DISCHARGE_FILE,
# next to this notebook; the table goes to the run log in the PDF.
DISCHARGE_FILE = 'lunar_discharge_table.py'
with open(DISCHARGE_FILE) as _f:
    exec(compile(_f.read(), DISCHARGE_FILE, 'exec'))

In [8]:
# Step 8
# Solar input (sun angle, heliostat cosine factor, receiver losses) and the radiator
# environment (lunar surface and sink temperatures) are in the file SOLAR_FILE, next to
# this notebook. This cell loads them; the quick-look table goes to the run log.
SOLAR_FILE = 'lunar_solar.py'
with open(SOLAR_FILE) as _f:
    exec(compile(_f.read(), SOLAR_FILE, 'exec'))

In [9]:
# Step 9
# Hour-by-hour simulation of the TES energy balance (day and night): simulate() is in the
# file SIMULATE_FILE, next to this notebook. This cell loads it; Step 10 uses it to size
# the plant and to run the final lunar cycle.
SIMULATE_FILE = 'lunar_simulate.py'
with open(SIMULATE_FILE) as _f:
    exec(compile(_f.read(), SIMULATE_FILE, 'exec'))

In [11]:
# Step 10
# ---------------------------------------------------------------------
# Sizing (Eq. 21-25 + collector area, Section VI.3), for each cycle
# Results are stored in R[ct] (a dictionary per cycle)
#   (a) Paper method (Eq. 21-24): night heat at the full-TES design point x night length,
#       over the full TES window T_TES_max -> T_TES_min (for comparison only)
#   (b) Smallest TES mass that carries the night alone: start full at sunset, no sun,
#       must stay above the usable floor (cutoff) for the whole night
#   (c) Design TES mass = TES_mass_margin x (b), for dawn and dusk
#   (d) Smallest collector that keeps the plant running cycle after cycle: start full at
#       sunset, run n_cycles_periodic cycles, the TES must never hit the floor and the
#       stored energy at the last sunset must not be lower than at the sunset before
#   (e) Final run: settle to the repeating state, then one full cycle from sunrise
# The whole sizing runs twice per cycle: pass 1 with a fixed gas-loop pressure drop gives
# the largest gas flow (m_gas^2/rho) of normal operation; that becomes the loop's design
# point, where dp = dp_loop * P_loop, and pass 2 sizes the plant with dp scaling from it.
# Radiator: Step 6 sized it for the worst night case down to the TES floor. After sizing it
# is resized to the largest area needed in normal operation (x rad_area_margin), so it is
# used ~100 % at its peak hour, and the final cycle is run again with it (the hours do not
# change: the radiator never limits the cycle in normal operation).
# Speed: pass 2 starts each search from pass 1's answer and steps out by bracket_step
# (x1.05 per try) until one size fails and one works, then bisects between them; pass 1
# (and any run without a guess) searches the full range. Same test, same tolerance.
# simulate(..., n_steps) runs n_steps + 1 steps (Step 9), so a period of n hours uses
# n_steps = n - 1. Bisections stop at a relative tolerance of size_rtol.
# ---------------------------------------------------------------------
size_rtol = 1e-2                       # relative tolerance of the mass and area bisections
bracket_step = 1.05                    # pass 2: step out from pass 1's answer by this factor

def _bisect_min(works, lo, hi, rtol=None):
    """Smallest x in [lo, hi] with works(x) True (works is monotonic: False below, True above)."""
    rtol = size_rtol if rtol is None else rtol
    while hi - lo > rtol * hi:
        mid = 0.5 * (lo + hi)
        lo, hi = (lo, mid) if works(mid) else (mid, hi)
    return hi

def _bracket(works, guess, x_min, x_max, step=None):
    """(lo, hi) around the answer with works(lo) False and works(hi) True, stepping out from
    guess by `step` per try; None if it leaves [x_min, x_max] (then search the full range)."""
    step = bracket_step if step is None else step
    x = guess
    if works(x):
        hi = x
        while True:
            lo = hi / step
            if lo < x_min:
                return None
            if not works(lo):
                return lo, hi
            hi = lo
    lo = x
    while True:
        hi = lo * step
        if hi > x_max:
            return None
        if works(hi):
            return lo, hi
        lo = hi

def size_system(ct, rtol=None, guess=None):
    """Size the plant for cycle ct. guess = pass-1 result (dict with m_TES_night, A_col):
    the searches then start from it instead of the full range."""
    # (a) Paper method: design-point Q_in for the whole night, full TES window
    p_design = plant_point(T_TES_max, ct=ct)
    E_req_design = p_design['Q_in'] * t_night                         # kWh, Eq. 21
    dE_window = (e_full - e_regolith(T_TES_min)) / 3.6e6              # kWh/kg
    m_TES_design = E_req_design / dE_window                           # Eq. 24

    # (b) Minimum TES mass for the night alone (start full at sunset, no sun)
    def survives_night(m):
        return simulate(m, 0.0, e_full, n_night - 1, t0=t_day, ct=ct)[0]
    br = _bracket(survives_night, guess['m_TES_night'], 0.1 * m_TES_design,
                  20 * m_TES_design) if guess else None
    if br is None:
        lo, hi = 0.1 * m_TES_design, 20 * m_TES_design
        if survives_night(lo) or not survives_night(hi):
            raise RuntimeError(f"{ct}: night-mass search range does not bracket the answer")
    else:
        lo, hi = br
    m_TES_night = _bisect_min(survives_night, lo, hi, rtol)

    # (c) Design TES mass with margin for dawn and dusk
    m_TES = TES_mass_margin * m_TES_night
    V_TES = m_TES / rho_TES                                           # Eq. 25

    # (d) Smallest sustainable collector area (TES must not run down cycle to cycle)
    N = n_cycles_periodic
    def survives_periodic(A):
        ok, out = simulate(m_TES, A, e_full, N * n_cycle, t0=t_day, ct=ct)
        if not ok:
            return False
        E = out['E_TES']                                              # index j*n_cycle = j-th sunset
        return E[N * n_cycle] >= E[(N - 1) * n_cycle] - 1e-6
    br = _bracket(survives_periodic, guess['A_col'], 1.0, 1e8) if guess else None
    if br is None:
        A_lo, A_hi = 0.0, 1e3
        while not survives_periodic(A_hi):
            A_hi *= 2
            if A_hi > 1e8:
                raise RuntimeError(f"{ct}: no collector area works, increase TES_mass_margin")
    else:
        A_lo, A_hi = br
    A_col = _bisect_min(survives_periodic, A_lo, A_hi, rtol)

    # (e) Final run: settle to the repeating state, then one cycle from sunrise
    ok_settle, pre = simulate(m_TES, A_col, e_full, (N - 1) * n_cycle + n_night - 1,
                              t0=t_day, ct=ct)                        # ends at sunrise
    survived, cycle = simulate(m_TES, A_col, pre['e_end'], n_cycle, t0=0.0, ct=ct)
    cycle.pop('e_end')
    day = {k: v[:n_day + 1] for k, v in cycle.items()}
    night = {k: v[n_day:] for k, v in cycle.items()}
    return dict(p_design=p_design, E_req_design=E_req_design, m_TES_design=m_TES_design,
                e_sunrise=pre['e_end'],
                m_TES_night=m_TES_night, m_TES=m_TES, V_TES=V_TES, A_col=A_col,
                cycle=cycle, day=day, night=night, survived=survived and ok_settle,
                periodic_error=cycle['T_TES'][-1] - cycle['T_TES'][0])

# --- Size each cycle: pass 1, pass 2, radiator to its peak, final cycle -----------------
# The cycles are independent, so with PARALLEL = True they are sized at the same time in
# separate processes (one per cycle, on separate processor cores). The results, and the
# order of the run-log lines, are exactly those of sizing them one after the other.
# Needs the 'fork' start method (Linux, Colab, macOS); on Windows -- or with
# PARALLEL = False -- the cycles are sized one after the other in this process.
PARALLEL = True

def _size_cycle(ct):
    """Full sizing of one cycle. Returns everything it sets, plus its run-log lines."""
    n_log = len(RUN_LOG)
    GAS_DP_REF[ct] = None
    R1 = size_system(ct, rtol=1e-2)                                   # pass 1: fixed loop dp (coarse)
    GAS_DP_REF[ct] = float(np.nanmax(R1['cycle']['K_gas']))          # loop design flow
    r = size_system(ct, guess=R1)                                     # pass 2: dp scales with flow,
                                                                      #   searches start from pass 1
    # radiator sized to the peak of normal operation, then the final cycle again
    cy = r['cycle']
    _on = cy['running'] & ~cy['part_load']
    A_cons = A_RAD[ct]
    A_RAD[ct] = rad_area_margin * float(np.nanmax(cy['A_need'][_on])) * (1 + 1e-9)
    ok_r, cy = simulate(r['m_TES'], r['A_col'], r['e_sunrise'], n_cycle, t0=0.0, ct=ct)
    cy.pop('e_end')
    r.update(cycle=cy, day={k: v[:n_day + 1] for k, v in cy.items()},
             night={k: v[n_day:] for k, v in cy.items()}, survived=r['survived'] and ok_r,
             periodic_error=cy['T_TES'][-1] - cy['T_TES'][0],
             p_design=plant_point(T_TES_max, ct=ct))
    # lowest TES temperature at which the plant still gives full power at night with it
    T_ok = T_TES_max
    for T in np.arange(T_TES_max, T_TES_min - 0.1, -2.0):
        p = plant_point(T, None, ct)
        if not p['running'] or p['part_load']:
            break
        T_ok = T
    E_sol = np.sum(r['cycle']['Q_solar'][:-1]) * dt
    E_cur = np.sum(r['cycle']['Q_curtail'][:-1]) * dt
    say(f"{ct:>11}: TES {r['m_TES']/1e3:7.1f} t ({r['V_TES']:5.0f} m3) | collector {r['A_col']:5.0f} m2 | "
        f"curtailed {100*E_cur/max(E_sol, 1e-9):4.1f}% | survives {r['survived']} | "
        f"repeats to {abs(r['periodic_error']):.3f} K")
    say(f"{'':>11}  gas loop design {r['cycle']['m_gas'][np.nanargmax(r['cycle']['K_gas'])]:.2f} kg/s "
        f"(dp {dp_loop*100:g} %) | radiator {A_cons:.0f} -> {A_RAD[ct]:.0f} m2, peak use "
        f"{np.nanmax(np.where(r['cycle']['running'], r['cycle']['rad_use'], np.nan))*100:.1f} %, "
        f"full power at night down to TES {T_ok:.0f} K")
    log = RUN_LOG[n_log:]
    del RUN_LOG[n_log:]                                  # added back in cycle order below
    return dict(ct=ct, R=r, A_RAD=A_RAD[ct], A_RAD_conservative=A_cons,
                GAS_DP_REF=GAS_DP_REF[ct], T_full=T_ok, log=log)

import multiprocessing as _mp, warnings as _warnings, time as _time
_t_size = _time.time()
_use_par = PARALLEL and len(cycles) > 1 and 'fork' in _mp.get_all_start_methods()
if _use_par:
    with _warnings.catch_warnings():
        _warnings.simplefilter('ignore', DeprecationWarning)   # fork from a threaded kernel
        with _mp.get_context('fork').Pool(len(cycles)) as _pool:
            _res = _pool.map(_size_cycle, cycles)
else:
    _res = [_size_cycle(ct) for ct in cycles]

R, A_RAD_conservative = {}, {}
for _r in _res:                                          # results back, in cycle order
    ct = _r['ct']
    R[ct] = _r['R']
    A_RAD[ct] = _r['A_RAD']
    A_RAD_conservative[ct] = _r['A_RAD_conservative']
    GAS_DP_REF[ct] = _r['GAS_DP_REF']
    CYC[ct]['T_TES_full_power'] = _r['T_full']
    RUN_LOG.extend(_r['log'])
say(f"Sizing took {(_time.time() - _t_size)/60:.1f} min "
    f"({'both cycles in parallel' if _use_par else 'one cycle after the other'})")

KeyboardInterrupt: 

In [ ]:
# Step 11
# --- Energy balances over one full lunar cycle, for each cycle (used by the report) ---
#   TES  : (solar - curtailed - drawn by cycle - insulation loss) - change in stored energy = 0
#   cycle: heat in - shaft power - heat out = 0, every hour the cycle runs
for ct in cycles:
    cy = R[ct]['cycle']
    _s = slice(0, -1)                                  # steps, not end point
    E_solar   = np.sum(cy['Q_solar'][_s]) * dt         # kWh
    E_curtail = np.sum(cy['Q_curtail'][_s]) * dt
    E_drawn   = np.sum(cy['Q_in'][_s]) * dt
    E_loss    = np.sum(cy['Q_loss'][_s]) * dt          # through the insulation
    E_change  = cy['E_TES'][-1] - cy['E_TES'][0]
    R[ct]['balance'] = (E_solar - E_curtail - E_drawn - E_loss) - E_change      # kWh, ~0
    _on = cy['running']                                # only hours the cycle runs
    R[ct]['cycle_balance'] = (np.max(np.abs(cy['Q_in'][_on] - cy['W_net'][_on] - cy['Q_out'][_on]))
                              if _on.any() else 0.0)  # kW, ~0
    say(f"{ct:>11}: TES balance {R[ct]['balance']:.2e} kWh, "
        f"cycle balance {R[ct]['cycle_balance']:.2e} kW")

In [ ]:
# Step 12
# Report, figures, references and the results PDF -- all in one go. The code is in the file
# REPORT_FILE, next to this notebook (Part A text report, Part B AIAA figures saved to
# figures_AIAA/, Part C references, Part D the single results PDF OUT_PDF).
# Each figure is rendered once and used for both its PNG file and its PDF page.
REPORT_FILE = 'lunar_report.py'
with open(REPORT_FILE) as _f:
    exec(compile(_f.read(), REPORT_FILE, 'exec'))